# ML Ops project
*Détecter les caractéristiques d'un visage*

DEFIVES François

GRANIER Sylvain

HACALA Maude

### 0. Vérification des prérequis

In [2]:
import sys
if '3.13.7' not in sys.version:
    print("Veuillez utiliser la version 3.13.7 de Python")
import pandas as pd
import torch
import pyspark
import time
import PIL
from PIL import Image
import numpy as np
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, col
from pyspark.sql.types import BinaryType, StringType
from PIL import Image
import numpy as np
import io
import os



### Import des données annotés 1

Chaque image annottée dans le dossier /data/lot1 est composée d'un fichier csv associé contenant ses caractéristiques.

### Description des données

### Traitement des images

Nous souhaitons redimensionner toutes les images à une taille de 64*64 et segmenter les visages et centrer sur le visage.

In [4]:
# Segmentation simple + redimensionnement des images du dossier
time_start=time.time()
input_folder = 'data/lot1_images/'
output_folder = 'data/lot1_resized/'
target_size = (64, 64)

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

for filename in os.listdir(input_folder):
    if not filename.lower().endswith('.png'):
        continue

    img_path = os.path.join(input_folder, filename)
    img = Image.open(img_path).convert("RGB")

    # Passage en niveaux de gris pour simplifier
    gray = img.convert("L")
    arr = np.array(gray)

    # Masque : pixels non blancs (ou pas trop blancs)
    # Tu peux ajuster le seuil 250 (0 = noir, 255 = blanc)
    mask = arr < 250

    # Si tout est blanc (image vide), on évite de planter
    if not mask.any():
        cropped = img
    else:
        # Coordonnées des pixels “utiles”
        ys, xs = np.where(mask)
        x_min, x_max = xs.min(), xs.max()
        y_min, y_max = ys.min(), ys.max()

        # Recadrage sur la zone utile
        cropped = img.crop((x_min, y_min, x_max + 1, y_max + 1))

    # Redimensionnement
    img_resized = cropped.resize(target_size)
    img_resized.save(os.path.join(output_folder, filename))
time_end=time.time()
print(f"Temps de traitement des images : {time_end - time_start} secondes")

Temps de traitement des images : 211.95414543151855 secondes


### Spark

### EN TRAIN DE MOURIR A CAUSE DE CE CONNARDS DE SPARK

### CNN

In [9]:
from torch.utils.data import Dataset
from PIL import Image
import pandas as pd
import os
import torch
import torchvision.transforms as T

class FaceDataset(Dataset):
    def __init__(self, img_dir, csv_path):
        self.img_dir = img_dir
        self.df = pd.read_csv(csv_path)

        self.transforms = T.Compose([
            T.ToTensor(),  # convertit en [0,1]
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.img_dir, row["filename"])
        img = Image.open(img_path).convert("RGB")
        img = self.transforms(img)

        return {
            "image": img,
            "barbe": torch.tensor(row["barbe"], dtype=torch.float32),
            "moustache": torch.tensor(row["moustache"], dtype=torch.float32),
            "lunette": torch.tensor(row["lunette"], dtype=torch.float32),
            "couleur": torch.tensor(row["couleur"], dtype=torch.long),
            "taille": torch.tensor(row["taille"], dtype=torch.long),
        }


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FaceAttributesCNN(nn.Module):
    """
    Sortie : logits de taille 11
    - [0] : barbe (binaire)
    - [1] : moustache (binaire)
    - [2] : lunette (binaire)
    - [3:8] : couleur cheveux (5 classes)
    - [8:11]: taille cheveux (3 classes)
    """
    def __init__(self):
        super().__init__()

        # Backbone convolutionnel
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),  # 64 -> 32

            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),  # 32 -> 16

            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),  # 16 -> 8

            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2),  # 8 -> 4
        )

        self.feature_dim = 256 * 4 * 4

        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 11)  # 3 binaires + 5 + 3
        )

    def forward(self, x):
        # x : (batch, 3, 64, 64)
        feat = self.backbone(x)
        logits = self.mlp(feat)  # (batch, 11)
        return logits


In [10]:
from torch.utils.data import DataLoader

dataset = FaceDataset(
    img_dir="data/lot1_resized",
    csv_path="data/labels.csv"
)

train_loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)


FileNotFoundError: [Errno 2] No such file or directory: 'data/labels.csv'

In [6]:
bce = nn.BCEWithLogitsLoss()
ce  = nn.CrossEntropyLoss()

def compute_loss(logits, y_barbe, y_moustache, y_lunette, y_couleur, y_taille):
    """
    logits : (batch, 11)
    labels :
      - y_barbe, y_moustache, y_lunette : (batch,) 0/1
      - y_couleur : (batch,) in [0..4]
      - y_taille  : (batch,) in [0..2]
    """
    # Découpage des sorties
    bin_logits    = logits[:, 0:3]   # barbe, moustache, lunette
    couleur_logits = logits[:, 3:8]  # 5 classes
    taille_logits  = logits[:, 8:11] # 3 classes

    # On empile les 3 binaires dans un seul tensor (batch, 3)
    y_bin = torch.stack([y_barbe, y_moustache, y_lunette], dim=1).float()

    loss_bin = bce(bin_logits, y_bin)          # multi-label binaire
    loss_col = ce(couleur_logits, y_couleur)  # multi-classe
    loss_tai = ce(taille_logits, y_taille)    # multi-classe

    # Moyenne des 3 pertes (tu pourras pondérer plus tard si besoin)
    loss = (loss_bin + loss_col + loss_tai) / 3.0
    return loss


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FaceAttributesCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

for epoch in range(10):
    model.train()
    running_loss = 0.0

    for batch in train_loader:
        images = batch["image"].to(device)  # (B, 3, 64, 64)
        y_barbe     = batch["barbe"].to(device)       # (B,)
        y_moustache = batch["moustache"].to(device)   # (B,)
        y_lunette   = batch["lunette"].to(device)     # (B,)
        y_couleur   = batch["couleur"].to(device)     # (B,)
        y_taille    = batch["taille"].to(device)      # (B,)

        optimizer.zero_grad()
        logits = model(images)  # (B, 11)

        loss = compute_loss(
            logits,
            y_barbe, y_moustache, y_lunette,
            y_couleur, y_taille
        )

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1} - loss: {epoch_loss:.4f}")


NameError: name 'train_loader' is not defined